Install Library

In [1]:
%pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 804.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 901.

Import Library

In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import mlflow
import mlflow.sklearn

# Scikit-learn datasets & model selection
from sklearn.datasets import( load_breast_cancer)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

#Evaluation matrics - THE CORE OF THIS NOTEBOOK
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    ConfusionMatrixDisplay
)

Load and Split Data

In [3]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train_c.shape}  Test: {X_test_c.shape}")

Train: (455, 30)  Test: (114, 30)


Configure MLflow tracking URI and experiment

In [6]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("breast-cancer-classifier")

2026/05/30 11:33:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/30 11:33:11 INFO mlflow.store.db.utils: Updating database tables
2026/05/30 11:33:14 INFO mlflow.tracking.fluent: Experiment with name 'breast-cancer-classifier' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/mlruns/1', creation_time=1780140794139, experiment_id='1', last_update_time=1780140794139, lifecycle_stage='active', name='breast-cancer-classifier', tags={}, trace_location=None, workspace='default'>

Manual Log

In [19]:
MAX_DEPTH = 4
MIN_SAMPLES = 5
CRITERION = 'gini'

mlflow.end_run()

with mlflow.start_run(run_name="manual-logging") as run :
  mlflow.log_param("max_depth", MAX_DEPTH)
  mlflow.log_param("min_sample_split", MIN_SAMPLES)
  mlflow.log_param("criterion", CRITERION)
  mlflow.log_param("random_state", 42 )

clf = DecisionTreeClassifier(
    max_depth=MAX_DEPTH,
    min_samples_split=MIN_SAMPLES,
    criterion=CRITERION,
    random_state=42
)
clf.fit(X_train_c, y_train_c)
y_pred = clf.predict(X_test_c)

#log Evaluation Matrics
mlflow.log_metric("accuracy", accuracy_score(y_test_c, y_pred))
mlflow.log_metric("precision", precision_score(y_test_c, y_pred))
mlflow.log_metric("recall", recall_score(y_test_c, y_pred))
mlflow.log_metric("f1", f1_score(y_test_c, y_pred))

#Save confusion matrix as a PNG artifact
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test_c, y_pred, ax=ax)
plt.tight_layout()
plt.savefig("confusion_matrix.png")
mlflow.log_artifact("confusion_matrix.png")
plt.close()

mlflow.sklearn.log_model(clf, "decision-tree-model")

print(f"RUN ID:   {run.info.run_id}")
print(f"Accuracy:  {accuracy_score(y_test_c, y_pred):.4f}")

mlflow.sklearn.autolog(
    log_input_examples = True,
    log_model_signatures = True,
    log_models = True
)
mlflow.end_run()
with mlflow.start_run(run_name="autolog-run"):
  clf2 = DecisionTreeClassifier(max_depth=6, random_state=42)
  clf2.fit(X_train_c, y_train_c)
  print("autolog run complete - check the UI for logged param & metrics")

!mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000 &
print("UI running at: http://localhost:5000")


2026/05/30 11:57:19 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'c1991bff9afa40b8892361f2fdf45ee9', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/30 11:57:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/30 11:57:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 11:57:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitra

RUN ID:   fc5380bd93734a98891aaad2262d5d3e
Accuracy:  0.9386
autolog run complete - check the UI for logged param & metrics
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
2026/05/30 11:57:47 INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
2026/05/30 11:57:47 INFO:     Started parent process [12283]
2026/05/30 11:58:24 INFO:     Started server process [12324]
2026/05/30 11:58:24 INFO:     Waiting for application startup.
2026/05/30 11:58:24 INFO:     Application startup complete.
2026/05/30 11:58:24 INFO:     Started server process [12325]
2026/05/30 11:58:24 INFO:     Waiting for application startup.
2026/05/30 11:58:24 INFO:     Application startup complete.
2026/05/30 11:58:24 INFO:     Started server process [12327]
2026/05/30 11:58:24 INFO:     Waiting for

Student Assigment

In [20]:
import mlflow, mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_tracking_uri("sqlite:///quiz.db")
mlflow.set_experiment("quiz-experiment")

data = load_iris()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

MAX_DEPTH = 5
N_ESTIMATORS = 100

with mlflow.start_run(run_name="quiz-run"):
  #1. Log Params
  mlflow.log_param("max_depth", MAX_DEPTH)
  mlflow.log_param("n_estimators", N_ESTIMATORS)

  model = RandomForestClassifier(
      max_depth = MAX_DEPTH, n_estimators=N_ESTIMATORS, random_state=42
  )
  model.fit(X_tr, y_tr)
  preds = model.predict(X_te)

  #2. Log Metrixs
  mlflow.log_metric("accuracy", accuracy_score(y_te, preds))
  mlflow.log_metric("f1", f1_score(y_te, preds, average='macro'))

  #3. Log and register model
  mlflow.sklearn.log_model(
      model,
      "random-forest-model",
      registered_model_name = "iris-rf-classifier"
  )

  #4. Set Tag
  mlflow.set_tag("team", "data-science")


2026/05/30 12:11:56 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/30 12:11:56 INFO mlflow.store.db.utils: Updating database tables
2026/05/30 12:11:58 INFO mlflow.tracking.fluent: Experiment with name 'quiz-experiment' does not exist. Creating a new experiment.
2026/05/30 12:11:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/30 12:12:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 12:12:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization me